# Índice FIBRAS de AMEFIBRA

Notebook para extraer la tabla pública del Índice FIBRAS. La página carga los datos dentro de un `iframe` mediante JavaScript y WebSocket, por lo que se utiliza Playwright con Chromium.

> La información se ofrece únicamente para consulta y análisis. AMEFIBRA indica que los datos tienen aproximadamente 20 minutos de retraso y no deben usarse como base única para decisiones de inversión.

In [1]:
import io
import re
import sys
import unicodedata
from datetime import datetime
from pathlib import Path
from typing import Optional

import pandas as pd
from playwright.sync_api import TimeoutError as PWTimeout
from playwright.sync_api import sync_playwright

In [2]:
URL_PAGINA = "https://amefibra.com/el-mercado/indice-fibras/"
PATRON_IFRAME_TABLA = re.compile(r"edimex\.com\.mx/Emisora/Reportes/?(\?.*)?$")
CARPETA_SALIDA = Path.cwd() / "output"
FUENTE_DATOS = "AMEFIBRA / Economatica México"

# Parámetros editables del notebook
HEADLESS = True
TIMEOUT_DATOS_MS = 30000
EXPORTAR_CSV_ANALITICO = True
EXPORTAR_CSV_EXCEL = False
EXPORTAR_XLSX = False
RUTA_CSV_EXCEL = Path.cwd() / "indice_fibras.csv"
RUTA_XLSX = Path.cwd() / "indice_fibras.xlsx"

## Funciones de extracción

In [3]:
def _limpiar_encabezado(texto: str) -> str:
    texto = re.sub(r"[↑↓]", "", str(texto))
    return re.sub(r"\s+", " ", texto).strip()


def _localizar_frame_tabla(page, intentos=10, espera_ms=1000):
    for _ in range(intentos):
        for frame in page.frames:
            if PATRON_IFRAME_TABLA.search(frame.url or ""):
                return frame
        page.wait_for_timeout(espera_ms)
    return None


def _extraer_html_tabla(frame) -> Optional[str]:
    return frame.evaluate("""() => {
        const tablas = Array.from(document.querySelectorAll('table'));
        let mejor = null, filasMax = -1;
        for (const tabla of tablas) {
            const filas = tabla.querySelectorAll('tbody tr').length;
            if (filas > filasMax) { filasMax = filas; mejor = tabla; }
        }
        return mejor ? mejor.outerHTML : null;
    }""")


def obtener_tabla_fibras(headless: bool = True, timeout_datos_ms: int = 30000) -> pd.DataFrame:
    with sync_playwright() as playwright:
        browser = playwright.chromium.launch(headless=headless)
        page = browser.new_page(locale="es-MX")
        try:
            page.goto(URL_PAGINA, wait_until="domcontentloaded")
            frame = _localizar_frame_tabla(page)
            if frame is None:
                raise RuntimeError("No se encontró el iframe con la tabla de FIBRAs.")
            try:
                frame.wait_for_function("""() => {
                    const filas = document.querySelectorAll('table tbody tr');
                    if (filas.length === 0) return false;
                    const celda = filas[0].querySelector('td:nth-child(2)');
                    const texto = celda ? celda.textContent.trim() : '';
                    return texto.length > 0 && texto !== '0' && texto !== '0.00';
                }""", timeout=timeout_datos_ms)
            except PWTimeout:
                print("Aviso: se agotó el tiempo esperando datos en vivo; se usará lo cargado.", file=sys.stderr)
            html_tabla = _extraer_html_tabla(frame)
        finally:
            browser.close()
    if not html_tabla:
        raise RuntimeError("No se pudo extraer la tabla de indicadores.")
    df = pd.read_html(io.StringIO(html_tabla))[0]
    df.columns = [_limpiar_encabezado(columna) for columna in df.columns]
    return df.dropna(axis=1, how="all")

## Normalización y exportación

In [4]:
def _a_snake_case(texto: str) -> str:
    texto = texto.replace("%", "pct")
    sin_acentos = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-zA-Z0-9]+", "_", sin_acentos).strip("_").lower()


def normalizar_para_analisis(df: pd.DataFrame, momento_extraccion: Optional[datetime] = None) -> pd.DataFrame:
    momento_extraccion = momento_extraccion or datetime.now()
    resultado = df.copy()
    resultado.columns = [_a_snake_case(str(columna)) for columna in resultado.columns]
    for columna in resultado.columns:
        serie = resultado[columna]
        if pd.api.types.is_string_dtype(serie):
            valores = serie.astype(str).str.strip()
            if valores.str.endswith("%").all():
                resultado[columna] = pd.to_numeric(valores.str.rstrip("%"), errors="coerce")
    resultado.insert(0, "fecha_hora_extraccion", momento_extraccion.isoformat(timespec="seconds"))
    resultado.insert(1, "fuente_datos", FUENTE_DATOS)
    return resultado


def exportar_csv_analitico(df: pd.DataFrame, carpeta_salida: Path = CARPETA_SALIDA) -> Path:
    momento = datetime.now()
    carpeta_salida.mkdir(parents=True, exist_ok=True)
    nombre = f"{momento:%Y%m%d_%H%M%S}_indice_fibras_amefibra.csv"
    ruta = carpeta_salida / nombre
    normalizar_para_analisis(df, momento).to_csv(ruta, index=False, encoding="utf-8")
    return ruta

## Ejecutar extracción

In [5]:
import asyncio
import concurrent.futures
import warnings

print(f"Consultando {URL_PAGINA} ...")


def _ejecutar_en_hilo_aparte():
    # El kernel de Jupyter ya corre un event loop de asyncio, y la API síncrona de
    # Playwright no admite ejecutarse dentro de uno (lanza Error), así que se
    # despacha a un hilo aparte. Playwright crea su propio loop internamente con
    # asyncio.new_event_loop(), que en Windows respeta la *policy* activa; el
    # kernel deja configurada una policy basada en SelectorEventLoop (para
    # compatibilidad con zmq/tornado), que no soporta subprocesos, y Playwright
    # necesita subprocesos para lanzar el navegador. Por eso se activa aquí,
    # solo para este hilo, la policy de Proactor que sí los soporta. La API de
    # policies está deprecada desde Python 3.14 (se retira en 3.16) pero sigue
    # siendo, por ahora, el único gancho disponible para influir en qué clase de
    # loop crea Playwright interceptar aquí (Playwright llama a
    # asyncio.new_event_loop() directo, sin exponer alternativa).
    if sys.platform == "win32":
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", DeprecationWarning)
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    return obtener_tabla_fibras(headless=HEADLESS, timeout_datos_ms=TIMEOUT_DATOS_MS)


with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
    df = executor.submit(_ejecutar_en_hilo_aparte).result()

print(f"Índice FIBRAS - {datetime.now():%Y-%m-%d %H:%M} (dato con ~20 min de retraso)")
display(df)

if EXPORTAR_CSV_ANALITICO:
    ruta_csv_analitico = exportar_csv_analitico(df)
    print(f"CSV analítico guardado en: {ruta_csv_analitico}")

Consultando https://amefibra.com/el-mercado/indice-fibras/ ...
Índice FIBRAS - 2026-08-22 17:05 (dato con ~20 min de retraso)


,Emisora,Cotización,Var.,Var. %,Apertura,Máx. día,Min. día,Promedio,Operaciones,Volumen,Importe,Máx. 52 s.,Min. 52 s.
0,DANHOS13,28.71,0.10,0.35%,28.77,28.75,29.00,28.54,2262,164212,4717301,29.12,23.65
1,EDUCA18,52.50,-1.50,-2.78%,52.50,52.50,52.50,52.50,11,366,19215,58.36,46.39
2,FIBRAMQ12,43.51,0.76,1.78%,43.05,42.95,43.95,42.16,1536,890429,38525363,45.27,27.73
3,FIBRAPL14,75.57,1.07,1.44%,75.43,74.80,76.06,74.80,4503,513731,38792854,83.97,64.05
4,FIBRAUP18,37.45,0.00,0.00%,37.45,37.45,37.45,37.45,11,32,1194,41.00,17.27
5,FIHO12,7.65,0.11,1.46%,7.59,7.65,7.66,7.53,236,23192,177352,8.01,6.92
6,FINN13,4.86,0.09,1.89%,4.83,4.77,4.88,4.77,179,8250,40058,5.40,4.33
7,FMTY14,14.50,0.28,1.97%,14.43,14.27,14.59,14.26,28161,7460899,107779291,15.79,12.32
8,FNOVA17,41.59,-0.13,-0.31%,42.16,42.01,42.80,41.52,369,13733,572960,45.95,27.00
9,FPLUS16,5.09,0.02,0.39%,5.05,5.10,5.10,5.01,130,10960,55246,6.00,4.82


CSV analítico guardado en: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260822_170548_indice_fibras_amefibra.csv


### Exportar a archivo de Excel - xlsx (Ejecución opcional)

In [6]:
if EXPORTAR_CSV_EXCEL:
    df.to_csv(RUTA_CSV_EXCEL, index=False, encoding="utf-8-sig")
    print(f"CSV compatible con Excel guardado en: {RUTA_CSV_EXCEL}")

if EXPORTAR_XLSX:
    df.to_excel(RUTA_XLSX, index=False)
    print(f"Excel guardado en: {RUTA_XLSX}")

## Solo emisoras

In [6]:
df_emisoras = df[["Emisora"]]
print(df_emisoras)

      Emisora
0    DANHOS13
1     EDUCA18
2   FIBRAMQ12
3   FIBRAPL14
4   FIBRAUP18
5      FIHO12
6      FINN13
7      FMTY14
8     FNOVA17
9     FPLUS16
10    FSHOP13
11     FUNO11
12     NEXT25
13     SOMA21
14  STORAGE18


## Historial de distribuciones por FIBRA

### Fuentes evaluadas

| Fuente | Cobertura BMV | Datos de distribuciones | Acceso y límites |
|---|---|---|---|
| Relación con Inversionistas del emisor | Sí, por emisora | Fuente primaria; puede incluir fechas, importe y componentes fiscales en PDF/XLSX | Gratuita, sin API uniforme; requiere localizar y procesar reportes de cada emisor |
| AMEFIBRA | Sí, índice agregado | Cotización e indicadores del índice; no publica aquí un histórico normalizado de distribuciones | Consulta web pública; no se expone una API de dividendos en esta tabla |
| BMV/BIVA | Sí | Información oficial de emisoras y eventos, según disponibilidad del portal | Consulta pública, pero sin una API gratuita y estable para este flujo |
| FMP, Alpha Vantage, Twelve Data, EODHD, Nasdaq Data Link y Polygon | Cobertura mexicana variable | La cobertura y profundidad de dividendos para tickers BMV no está garantizada en el plan gratuito | Requieren revisar ticker, API key y límites por proveedor |
| `yfinance` | Sí para tickers Yahoo con sufijo `.MX`, cuando Yahoo dispone del evento | Fecha ex-dividendo y monto; no garantiza fecha de registro, pago ni componentes fiscales | Gratis y sin API key, pero es un cliente no oficial de Yahoo Finance y está sujeto a cambios y límites |

Se usa `yfinance` como respaldo reproducible porque las fuentes primarias no ofrecen una API homogénea. El resultado contiene la fecha ex-dividendo y el importe disponible en Yahoo; la fecha de registro, fecha de pago y componentes fiscales no se incluyen porque esta fuente no los entrega de forma confiable. El histórico se ordena del más antiguo al más reciente. `yield_pct` es el rendimiento de cada distribución respecto al cierre de su fecha ex-dividendo; `annualized_yield_pct` anualiza ese rendimiento usando `365 / días_del_periodo`. Para la primera fila se usa la mediana histórica de días entre distribuciones.

In [14]:
import time

import yfinance as yf

COLUMNAS_DISTRIBUCIONES = [
    "ticker",
    "ex_date",
    "amount_mxn",
    "close_on_ex_date_mxn",
    "yield_pct",
    "annualized_yield_pct",
    "periodicity",
]


def _normalizar_ticker(ticker: str) -> str:
    ticker = str(ticker).strip().upper()
    if not ticker:
        raise ValueError("El ticker no puede estar vacío.")
    return ticker.removesuffix(".MX")


def _detectar_periodicidad(fechas: pd.Series) -> str:
    diferencias = fechas.sort_values().diff().dt.days.dropna()
    if diferencias.empty:
        return "indeterminada"
    mediana = diferencias.median()
    if mediana <= 45:
        return "mensual"
    if mediana <= 120:
        return "trimestral"
    return "otra"


def _descargar_con_reintentos(ticker_yahoo: str, intentos: int = 3, espera_s: float = 2.0):
    ultimo_error = None
    for intento in range(intentos):
        try:
            return yf.Ticker(ticker_yahoo).dividends
        except Exception as error:
            ultimo_error = error
            if intento < intentos - 1:
                time.sleep(espera_s * (intento + 1))
    raise RuntimeError(f"No se pudo descargar distribuciones de {ticker_yahoo}.") from ultimo_error


def obtener_distribuciones(ticker: str, intentos: int = 3) -> pd.DataFrame:
    """Obtiene distribuciones históricas de una FIBRA BMV y las exporta a output/."""
    ticker_base = _normalizar_ticker(ticker)
    ticker_yahoo = f"{ticker_base}.MX"
    serie = _descargar_con_reintentos(ticker_yahoo, intentos=intentos)
    if serie.empty:
        raise ValueError(f"Yahoo Finance no devolvió distribuciones para {ticker_yahoo}.")

    distribuciones = serie.rename("amount_mxn").rename_axis("ex_date").reset_index()
    distribuciones["ex_date"] = pd.to_datetime(distribuciones["ex_date"], errors="coerce").dt.tz_localize(None).dt.normalize()
    distribuciones["amount_mxn"] = pd.to_numeric(distribuciones["amount_mxn"], errors="coerce")
    distribuciones = distribuciones.dropna(subset=["ex_date", "amount_mxn"])
    distribuciones = distribuciones[distribuciones["amount_mxn"] > 0]
    distribuciones = distribuciones.drop_duplicates(subset=["ex_date", "amount_mxn"])

    precios = yf.download(
        ticker_yahoo,
        start=(distribuciones["ex_date"].min() - pd.Timedelta(days=5)).strftime("%Y-%m-%d"),
        end=(distribuciones["ex_date"].max() + pd.Timedelta(days=2)).strftime("%Y-%m-%d"),
        auto_adjust=False,
        progress=False,
        threads=False,
    )
    if precios is None or precios.empty:
        cierre = pd.Series(dtype=float)
    else:
        if isinstance(precios.columns, pd.MultiIndex):
            precios.columns = precios.columns.get_level_values(0)
        cierre = precios["Close"].copy() if "Close" in precios else pd.Series(dtype=float)
        cierre.index = pd.to_datetime(cierre.index).tz_localize(None).normalize()

    distribuciones["close_on_ex_date_mxn"] = distribuciones["ex_date"].map(cierre)
    distribuciones["yield_pct"] = (
        distribuciones["amount_mxn"] / distribuciones["close_on_ex_date_mxn"] * 100
    ).where(distribuciones["close_on_ex_date_mxn"] > 0)
    diferencias_dias = distribuciones["ex_date"].sort_values().diff().dt.days
    intervalo_referencia = diferencias_dias.dropna().median()
    distribuciones["annualized_yield_pct"] = (
        distribuciones["yield_pct"] * 365 / diferencias_dias.fillna(intervalo_referencia)
    )
    periodicidad = _detectar_periodicidad(distribuciones["ex_date"])
    resultado = pd.DataFrame({
        "ticker": ticker_base,
        "ex_date": distribuciones["ex_date"],
        "amount_mxn": distribuciones["amount_mxn"],
        "close_on_ex_date_mxn": distribuciones["close_on_ex_date_mxn"],
        "yield_pct": distribuciones["yield_pct"],
        "annualized_yield_pct": distribuciones["annualized_yield_pct"],
        "periodicity": periodicidad,
    })
    resultado = resultado.sort_values("ex_date").reset_index(drop=True)[COLUMNAS_DISTRIBUCIONES]
    resultado["ex_date"] = resultado["ex_date"].dt.strftime("%Y-%m-%d")
    momento = datetime.now()
    CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
    ruta = CARPETA_SALIDA / f"{momento:%Y%m%d_%H%M%S}_{ticker_base}_dividendos.csv"
    resultado.to_csv(ruta, index=False, encoding="utf-8")
    resultado.attrs["ruta_csv"] = ruta
    return resultado

In [15]:
TICKER_PRUEBA = "FUNO11"
assert TICKER_PRUEBA in set(df_emisoras["Emisora"]), "El ticker de prueba no está en el listado AMEFIBRA."

historial_dividendos = obtener_distribuciones(TICKER_PRUEBA)
assert list(historial_dividendos.columns) == COLUMNAS_DISTRIBUCIONES
assert historial_dividendos["ex_date"].is_monotonic_increasing
assert not historial_dividendos.duplicated(subset=["ticker", "ex_date", "amount_mxn"]).any()
assert (historial_dividendos["amount_mxn"] > 0).all()
assert historial_dividendos["annualized_yield_pct"].notna().all()
assert Path(historial_dividendos.attrs["ruta_csv"]).exists()

print(f"Ticker probado: {TICKER_PRUEBA}. Registros: {len(historial_dividendos)}")
print(f"Periodicidad detectada: {historial_dividendos['periodicity'].iloc[0]}")
print(f"CSV generado: {historial_dividendos.attrs['ruta_csv']}")
display(historial_dividendos.tail(10))

Ticker probado: FUNO11. Registros: 64
Periodicidad detectada: trimestral
CSV generado: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260822_171759_FUNO11_dividendos.csv


,ticker,ex_date,amount_mxn,close_on_ex_date_mxn,yield_pct,annualized_yield_pct,periodicity
54,FUNO11,2024-05-07,0.402963,25.280001,1.593999,10.578358,trimestral
55,FUNO11,2024-08-08,0.519023,24.420000,2.125401,8.341629,trimestral
56,FUNO11,2024-11-08,0.525000,22.990000,2.283602,9.059941,trimestral
57,FUNO11,2025-02-07,0.551328,22.139999,2.490190,9.988124,trimestral
58,FUNO11,2025-05-08,0.554953,25.010000,2.218924,8.998971,trimestral
59,FUNO11,2025-08-08,0.570000,26.209999,2.174743,8.628055,trimestral
60,FUNO11,2025-11-07,0.605000,28.040001,2.157632,8.654238,trimestral
61,FUNO11,2026-02-06,0.670000,28.780001,2.328006,9.337604,trimestral
62,FUNO11,2026-05-08,0.620000,30.180000,2.054341,8.239938,trimestral
63,FUNO11,2026-08-07,0.639779,30.340000,2.108698,8.457965,trimestral
